# ML-03 — Frame Ranking Signal Analysis as an ML Task

[Open in Colab](https://colab.research.google.com/github/colkeaz/MLInternship/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This continues the Ranking Signal Analysis lane selected in Week 1. The framing below is an AI-assisted draft for me to review and explain in my own words before submission. No model is trained in this framing assignment.

To rerun locally, install `requirements.txt` plus `nbformat nbclient ipykernel`, then run all cells with the repository or this notebook's directory as the working directory. The bundled starter CSV is the only data source.


## 1. My lane as an ML task (type)

**Lane: Ranking Signal Analysis. Task type: scoring**, implemented later as regression of observed average search position. A smaller predicted value represents better observed positioning. The lane name does not mean this is a search-engine ranking algorithm or an editorial priority queue.

For content creators and strategists deciding where to focus SEO effort, I will examine whether content length, age, freshness, intent, and keyword context are associated with average search position. The output will be a signal report with grouped comparisons and, if justified, a simple scoring model evaluated by mean absolute error (MAE). Strategists can use the report to select comparable pages for a manual content/intent review and propose a small test, rather than automatically lengthening every article. A false signal wastes editing time and can damage useful content.

**ML loop:** choose the decision → load and check page-level data → define observed target and candidate features → establish a fixed baseline → later fit and validate on held-out clients → review associations with a strategist → test a content change and monitor new outcomes. This notebook covers framing and the baseline, not proof of a useful model.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error

root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data/raw/content_refresh_anonymized.csv").is_file()), None)
if root is None:
    raise FileNotFoundError("Run from this repository or work/notebooks with the starter CSV present.")

# An explicit allowlist keeps unrelated fields out of both analysis and outputs.
numeric_features = ["word_count", "content_age_days", "days_since_last_update",
                    "search_volume", "competition", "cpc"]
categorical_features = ["content_type", "main_intent"]
columns = ["content_id", "client_id", "avg_position", *numeric_features, *categorical_features]
raw = pd.read_csv(root / "data/raw/content_refresh_anonymized.csv", usecols=columns)
assert raw["content_id"].notna().all() and raw["content_id"].is_unique
assert raw["client_id"].notna().all()
lane = raw.loc[raw["avg_position"].gt(0)].copy()
assert not lane.empty
display(pd.DataFrame({"check": ["Starter pages", "Selected columns", "Pages without positive position",
                                "Lane pages with measured position", "Lane client groups"],
                      "value": [len(raw), len(columns), len(raw) - len(lane),
                                len(lane), lane["client_id"].nunique()]}))


,check,value
0,Starter pages,30000
1,Selected columns,11
2,Pages without positive position,1205
3,Lane pages with measured position,28795
4,Lane client groups,31


## 2. Target or proxy

**Target column: `target_avg_position_90d`**, copied from the measured `avg_position`. It is a continuous observed outcome over the trailing 90-day window, and a proxy for search positioning, not content quality, business value, or refresh benefit. Lower is better. A page average mixes queries and days and can hide very different search experiences.

`avg_position = 0` means missing measurement; those rows are excluded instead of being labeled excellent. There is no top-10 threshold label, product score, or decline-rule label to imitate. The target is shown as actual values below, not invented future outcomes.

This is a **same-snapshot association task**, not a forecast: metadata is recorded at export time and may have changed during the outcome window. For future prediction I would need timestamped features strictly before a later outcome window. The starter slice cannot establish that ordering. Excluding unmeasured pages also limits conclusions to measured pages in this starter sample.


In [2]:
target = "target_avg_position_90d"
lane[target] = lane["avg_position"].astype(float)
assert np.isfinite(lane[target]).all() and lane[target].gt(0).all()
assert lane[target].equals(lane["avg_position"].astype(float))

# Five safe numeric examples show the target shape; no identifiers are displayed.
display(lane[["avg_position", target]].head(5).reset_index(drop=True))
display(lane[target].describe().to_frame("measured position"))


,avg_position,target_avg_position_90d
0,10.6,10.6
1,20.3,20.3
2,36.5,36.5
3,6.2,6.2
4,44.0,44.0


,measured position
count,28795.000000
mean,17.026268
std,15.152439
min,0.100000
25%,6.700000
50%,11.400000
75%,22.900000
max,245.000000


## 3. Success metric

**Primary metric: MAE in average-position units; lower is better.** MAE is the mean absolute difference between predicted and measured position. It measures how close a continuous score is to the observed target and is easier to interpret than squared error. It does not measure the effect of editing a page or the quality of a ranked action queue.

The fixed baseline predicts the **training pages' median position** for every validation page. The code computes that baseline now. A provisional development goal is at least **10% lower MAE** than this baseline on the same held-out client groups; this is a chosen bar, not an established business threshold. No improvement is claimed here.

I use a reproducible 25% client-group holdout (seed 42), keeping all pages from a client together. This tests transfer to other client groups, not future performance. Later model selection should use additional grouped development folds and reserve a separate final test; this split is a development check. Report per-client errors and coverage as well as pooled MAE so large clients cannot silently dominate the conclusion. A scoring improvement alone does not establish an actionable content signal.


In [3]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, valid_idx = next(splitter.split(lane, groups=lane["client_id"]))
train, valid = lane.iloc[train_idx], lane.iloc[valid_idx]
assert set(train["client_id"]).isdisjoint(set(valid["client_id"]))
baseline_prediction = float(train[target].median())
baseline_mae = mean_absolute_error(valid[target], np.full(len(valid), baseline_prediction))
display(pd.DataFrame({"measure": ["Training pages", "Validation pages", "Training clients",
                                  "Validation clients", "Training median position",
                                  "Baseline validation MAE", "Provisional model MAE ceiling"],
                      "value": [len(train), len(valid), train["client_id"].nunique(),
                                valid["client_id"].nunique(), baseline_prediction,
                                baseline_mae, baseline_mae * 0.90]}))


,measure,value
0,Training pages,22024.000000
1,Validation pages,6771.000000
2,Training clients,23.000000
3,Validation clients,8.000000
4,Training median position,12.700000
5,Baseline validation MAE,8.551809
6,Provisional model MAE ceiling,7.696628


## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content item (page) with a positive measured average position in the starter export's trailing 90-day window.** This is not one query, one daily observation, or one client. The uniqueness check in section 1 verifies the page grain. The lane includes all pages with measured position rather than applying a favorable-performance filter.

The dataframe below contains candidate features plus the target. Identifiers remain available privately for grain checks and client-group splits, but are excluded from the feature matrix and displayed preview. Missing keyword context and word count stay missing, with explicit availability flags; zero must not silently mean “unmeasured.” Any later imputation and categorical encoding must be fitted on training groups only.

Excluded from candidate features: `avg_position`, `position_tier`, and the copied target (direct answer leakage); trend fields and derived decline labels; IDs; provider/model identifiers; and same-window traffic, CTR, and engagement metrics, which are entangled with position. These are candidate associations only, not certified pre-outcome predictors. CTR and other rate fields are percentages in this dataset, but this framing does not use them.


In [4]:
X = lane[numeric_features + categorical_features].copy()
for column in numeric_features + categorical_features:
    X[f"has_{column}"] = X[column].notna().astype("int8")
analysis_df = X.join(lane[[target]])
assert len(analysis_df) == len(lane)
assert not {"content_id", "client_id", "avg_position", "position_tier", target}.intersection(X.columns)
assert not X.columns.str.contains("trend|declining").any()
display(analysis_df.head(5).reset_index(drop=True))
display(X[numeric_features + categorical_features].isna().mean().mul(100)
        .rename("missing_percent").to_frame().round(2))


,word_count,content_age_days,days_since_last_update,search_volume,competition,cpc,content_type,main_intent,has_word_count,has_content_age_days,has_days_since_last_update,has_search_volume,has_competition,has_cpc,has_content_type,has_main_intent,target_avg_position_90d
0,3221.0,187,20,10.0,0.67,2.05,keyword article,transactional,1,1,1,1,1,1,1,1,10.6
1,2481.0,445,25,90.0,0.01,0.05,keyword article,informational,1,1,1,1,1,1,1,1,20.3
2,3515.0,141,20,0.0,0.00,0.00,keyword article,informational,1,1,1,1,1,1,1,1,36.5
3,NaN,463,22,10.0,0.00,0.00,keyword article,commercial,0,1,1,1,1,1,1,1,6.2
4,2803.0,263,14,0.0,0.00,0.00,keyword article,informational,1,1,1,1,1,1,1,1,44.0


,missing_percent
word_count,26.69
content_age_days,0.00
days_since_last_update,0.00
search_volume,5.97
competition,5.97
cpc,5.97
content_type,0.00
main_intent,5.64


## 5. Why ML beats a fixed rule here

**It has not been shown to beat a rule yet.** A rule such as “longer articles rank better” ignores intent, content type, keyword competition, age, and differences between clients. These may interact, and a model could summarize their joint association better than one cutoff. The grouped table below checks whether content types differ in observed position and measurement coverage; it does not estimate an effect of word count.

I would start with grouped comparisons, then consider a simple interpretable regression/scoring model only if it improves the declared baseline on unseen client groups. If it does not, I would retain the baseline or a descriptive report. Even improved prediction would not prove that changing a feature changes rankings: client mix, demand, competition, selection, and timing can confound associations.

**Action from the output:** use a stable, contextual signal to choose a small set of comparable pages for a strategist's review of content completeness and intent fit. Investigate before editing, test changes on a limited set, and monitor later search outcomes. Do not automatically add words, prune a page, or promise traffic gains from a model score.


In [5]:
# Aggregates only: a descriptive check, not a model or a causal comparison.
profile = lane.groupby("content_type", dropna=False).agg(
    pages=(target, "size"),
    median_observed_position=(target, "median"),
    word_count_coverage=("word_count", lambda s: s.notna().mean()),
    keyword_volume_coverage=("search_volume", lambda s: s.notna().mean()),
)
display(profile.round(3))
assert int(profile["pages"].sum()) == len(lane)
print("Framing checks passed. Baseline measured; no trained-model improvement or causal effect claimed.")


,pages,median_observed_position,word_count_coverage,keyword_volume_coverage
content_type,,,,
comparison article,697,8.5,1.000,1.000
feedly article,1366,6.4,1.000,0.000
keyword article,26732,12.0,0.712,0.987


Framing checks passed. Baseline measured; no trained-model improvement or causal effect claimed.


### Intern Review

**1. Task Type & ML Loop Mapping**

I frame Ranking Signal Analysis as a scoring task using regression to predict continuous search ranking position rather than building an algorithmic search engine or an automated editorial queue. In the ML loop, we start with the editorial decision to prioritize content updates, ingest the starter dataset, filter to pages with positive measured rank, and map content and market signals to an observed outcome. The model is evaluated against a fixed median baseline using group-held-out validation across client domains so that content strategists can triage underperforming outlier pages for manual review, after which the team monitors trailing 90-day rank changes from controlled editorial tests.   


**2. Unit of Analysis & Target Proxy**

The unit of analysis is one published content page (content_id) with a positive measured average position in the trailing 90-day window, grouped hierarchically by client domain (client_id). The target column is target_avg_position_90d (a continuous float where lower values indicate better search visibility), which serves as an observed proxy for search performance rather than direct content quality, business revenue, or guaranteed refresh impact. Because this export is a single static snapshot, it captures same-snapshot statistical associations rather than forward-looking causal predictions, and pages with avg_position <= 0 are excluded to prevent unmeasured pages from corrupting the target distribution.   


**3. Success Metric & Baseline**

The primary evaluation metric is Mean Absolute Error (MAE) in average-position units, which evaluates prediction deviation linearly without heavily penalizing long-tail outliers like squared error would. Using a reproducible 25% client-group split (GroupShuffleSplit, seed 42) to test generalization across unseen domains, the naive baseline predicts the training set median position of 12.70, yielding a validation MAE of 8.55. To justify moving forward with machine learning over the next 7 weeks, any candidate scoring model must clear a provisional success ceiling of at least a 10% error reduction, requiring an MAE of 7.70 or lower on held-out client groups.   


**4. Action Supported & Cost of Errors**

The scoring output serves as an operational triage filter for SEO strategists and editors to benchmark pages against comparable peers and identify genuine underperformers. Rather than triggering automated content alterations, the output guides strategists to inspect specific outliers for intent mismatch or topical gaps before launching small, controlled editorial experiments. The cost of a false positive is severe: content teams waste hundreds of editorial hours artificially lengthening articles, potentially diluting concise, high-ranking pages and hurting client revenue; conversely, a false negative leaves decaying pages unaddressed, leading to silent traffic attrition.   


**5. Why ML Beats a Fixed Heuristic**

A fixed editorial heuristic—such as requiring every article to surpass a specific word count or update threshold—fails because content and keyword attributes interact non-linearly across formats and intents. In the starter slice of 28,795 measured pages across 31 clients, feedly articles achieve a median position of 6.4 despite having no keyword volume data, comparison articles average 8.5, and keyword articles sit at 12.0 while exhibiting a 26.69% missingness rate in word counts. Machine learning beats fixed heuristics by learning these multivariate interactions across content types, search volume, and intent while explicitly modeling feature missingness flags rather than collapsing under broad, one-size-fits-all editorial rules.   


## Self-check

- [x] Named the lane, scoring task, observed target/proxy, metric, and provisional success bar.
- [x] Loaded the starter lane and checked that one row represents one page.
- [x] Displayed a real feature dataframe and the actual target column.
- [x] Computed a fixed baseline using disjoint client groups.
- [x] Explained the content decision, cost of an error, and limits of the snapshot.
- [x] Notebook executed top to bottom with outputs and no errors.
- [x] Outputs contain only safe numeric examples and aggregates, with no client names or raw queries.
- [x] Intern review: explain the framing in my own words and confirm it matches my intended question before submitting.

Applied repository skills: [framing-ml-problems](../../skills/framing-ml-problems/SKILL.md) and [flyrank-data](../../skills/flyrank/flyrank-data/SKILL.md). Column meanings and sentinel handling follow the [data dictionary](../../docs/data-dictionary.md).

Submission after review: commit this executed notebook under `work/notebooks/`, then submit the repository URL on the assignment card.
